## SQL Hospital Supply Chain Analysis

### Executive Summary

This analysis uncovers key insights in hospital supply chain and operations using SQL on real-world datasets:

- **Inventory Risks:** Critical items like ventilators and surgical masks may run out before restock due to vendor lead times exceeding stock availability.
- **Cost Drivers:** Staffing and supplies are largest expense categories, with noticeable monthly spikes indicating seasonal demand.
- **Patient Care Staffing:** MRI and appendectomy procedures require longer stays and more staff, highlighting areas for improved resource allocation.
- **Vendor Concerns:** Vendor EquipMed Co. supplies high-cost, critical items with the longest lead times, posing operational risks.
- **Operational Inefficiencies:** High equipment usage and staff overtime suggest potential bottlenecks and workload imbalances.

---

### Summary Recommendations

- Increase buffer stock for high-risk inventory items.
- Monitor and control spending around peak months in supplies and staffing.
- Allocate staffing efficiently for complex procedures to reduce stay lengths.
- Prioritize renegotiation or diversification for slow vendors supplying critical items.
- Review staff scheduling and equipment maintenance to optimize operations

#### enable SQLmagic
!pip install ipython-sql pandas


In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('healthcare_supply_chain.db')
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';", 
    conn
)
print("Imported Tables:")
print(tables)


Imported Tables:
             name
0  financial_data
1  inventory_data
2    patient_data
3      staff_data
4     vendor_data


In [3]:
schema = pd.read_sql_query("PRAGMA table_info(vendor_data);", conn)
print("inventory_data Schema:")
print(schema)

inventory_data Schema:
   cid                  name     type  notnull dflt_value  pk
0    0             Vendor_ID     TEXT        0       None   0
1    1           Vendor_Name     TEXT        0       None   0
2    2         Item_Supplied     TEXT        0       None   0
3    3  Avg_Lead_Time (days)  INTEGER        0       None   0
4    4         Cost_Per_Item     REAL        0       None   0
5    5       Last_Order_Date     TEXT        0       None   0
6    6    Next_Delivery_Date     TEXT        0       None   0


### Business Question 1: How can supply chain managers optimize inventory to prevent critical stockouts?

**Why this matters:**  
Hospital operations depend on having key supplies available. Stockouts cause delays in care and can affect patient outcomes. We want to identify items at risk of running out before restock arrives, so managers can prioritize orders.


In [31]:
query = """
with shortage as(
SELECT i.Date, i.Item_ID, i.Item_Name, i.Restock_Lead_Time as "restock_time", v.Vendor_Name,
round(1.0*Current_Stock / nullif(Avg_Usage_Per_Day,0),1) as days_stock_left
from inventory_data i 
left join vendor_data v on i.Vendor_ID = v.Vendor_ID
) 

select *
from shortage
where days_stock_left < restock_time
order by days_stock_left 
"""

In [60]:
result = pd.read_sql_query(query, conn)
print(result.head(10))

         Date  Item_ID      Item_Name  restock_time       Vendor_Name  \
0  2025-02-05      100  Surgical Mask            14      EquipMed Co.   
1  2025-01-22      103  X-ray Machine            19      EquipMed Co.   
2  2025-07-21      101         Gloves            24  MedSupplies Inc.   
3  2025-10-27      105  X-ray Machine             2      EquipMed Co.   
4  2026-01-05      107  Surgical Mask            17  HealthTools Ltd.   
5  2025-05-30      104  X-ray Machine             4      EquipMed Co.   
6  2025-06-14      105     Ventilator            15  MedSupplies Inc.   
7  2024-10-08      105        IV Drip            11      EquipMed Co.   
8  2024-10-21      105  X-ray Machine            17      EquipMed Co.   
9  2025-08-08      100     Ventilator             8  MedSupplies Inc.   

   days_stock_left  
0              0.2  
1              0.3  
2              0.3  
3              0.3  
4              0.3  
5              0.4  
6              0.4  
7              0.5  
8      

Multiple critical items, including surgical masks and ventilators, show dangerously low days of stock remaining compared to their vendor restock times. For example, several entries have less than 1 day of stock left while vendor lead times range from 14 to 29 days.

**Key Insight**:
Items with days_stock_left less than their restock_time indicate a high risk of stockout, potentially disrupting hospital operations and patient care.

**Business Suggestion:**

- Increase stock for high-risk items to cover lead times.  

- Negotiate faster delivery schedules or diversify vendors to reduce dependency on slow suppliers.  

- Implement continuous monitoring alerts for stock levels relative to lead times.

### Business Question 2:
Which expense categories have the greatest impact on hospital financial health over time?

#### Why this matters:
Understanding expense patterns and identifying which categories (Staffing, Supplies, Equipment, etc.) most affect the hospital’s finances helps management control costs, improve budgeting, and spot unusual spikes/ changes or regular seasonal trends.

In [33]:
query2 = """ 
select Expense_Category, sum(Amount) as total_spent
from financial_data
group by Expense_Category
order by total_spent desc
"""

In [61]:
query2_monthlytrend = """ 
select strftime('%Y/%m', Date)as Months, Expense_Category, sum(Amount) as total_spent
from financial_data
group by Months, Expense_Category
order by Months, total_spent desc
"""
monthly_expenses = pd.read_sql_query(query2_monthlytrend, conn)
print(monthly_expenses.head(10))

    Months Expense_Category  total_spent
0  2024/10         Supplies    296244.02
1  2024/10         Staffing    257058.47
2  2024/10        Equipment    214084.75
3  2024/11        Equipment    368917.30
4  2024/11         Supplies    176533.83
5  2024/11         Staffing    120044.67
6  2024/12         Supplies    457924.63
7  2024/12         Staffing    219526.60
8  2024/12        Equipment    100171.56
9  2025/01         Staffing    336755.71


Supplies and staffing consistently represent the largest portions of hospital expenses monthly. For example, in October 2024, supplies expenses reached nearly 300K, closely followed by staffing at around 257K. Noticeable spikes, such as the nearly $458K spent on supplies in December 2024, suggest seasonal or situational demand increases.  

**Key Insight:**
Supplies and staffing are the primary cost drivers, with monthly expense peaks indicating periods requiring closer financial oversight.

**Business Suggestion:**

- Focus budgeting and procurement efforts on these categories, especially during identified high-spend months.

- Implement tighter cost controls and inventory planning in peak seasons to manage cash flow and avoid overspending.

### Business Question 3:
Can we link patient care (outcomes, procedures, length of stay) to staffing or supply/inventory issues?

#### Why this matters:
Hospital efficiency and patient outcomes depend on having the right staff and supplies for each procedure. If staffing is low or the right equipment is unavailable, it may increase length of stay or complicate care. Identifying such patterns can reveal risks and opportunities for process improvement.

In [40]:
query3 = """ 
select Procedure_Performed, avg(Bed_Days) as stay_period,
avg(cast(
        (case 
            when instr(p.Staff_Needed, ',') > 0 
            then (length(p.Staff_Needed) - length(replace(p.Staff_Needed, ',', '')) + 1)
            else 1
        end) as integer)) as Avg_Staff_Needed
from patient_data p
group by Procedure_Performed
order by stay_period desc

"""
care_trends = pd.read_sql_query(query3, conn)
print(care_trends)

  Procedure_Performed  stay_period  Avg_Staff_Needed
0                 MRI     7.969466          1.358779
1        Appendectomy     7.435115          1.343511
2          Blood Test     7.130000          1.310000
3         Chest X-ray     6.891304          1.326087


Procedures like MRI and Appendectomy have the longest average patient stays (around 7.4 to 8 days) and require more staff on average (approximately 1.3 to 1.36 staff members per procedure). Blood Tests and Chest X-rays involve slightly shorter stays but still show a consistent staffing need above one.

**Key Insight:**
Longer, more complex procedures correlate with higher staff requirements, indicating these procedures demand more resources and careful scheduling.

**Business Suggestion:**

- Optimize staffing allocation to ensure sufficient resource availability for high-demand procedures like MRIs and Appendectomies.

- Explore ways to reduce patient stays for these procedures through process improvements and resource efficiency to improve hospital throughput.


### Business Question 4:
Which vendors have caused the most delays or cost overruns, especially on critical equipment?

#### Why this matters:
Vendor reliability directly impacts hospital operations. Delayed deliveries or cost overruns on essential equipment can disrupt patient care and strain budgets. Identifying vendors with poor performance helps focus negotiation and procurement efforts.

In [51]:
query4_updated = """
select v.Vendor_ID, v.Vendor_Name, v.Item_Supplied, v."Avg_Lead_Time (days)" as avg_leadtime,
    v.Cost_Per_Item, 
    count(i.Item_ID) as Items_Supplied_Count,
    sum(i.Current_Stock * v.Cost_Per_Item) as Total_Stock_Value,
    min(v.Next_Delivery_Date) as Next_Delivery_Date
FROM vendor_data v
left join inventory_data i on v.Vendor_ID = i.Vendor_ID
group by v.Vendor_ID, v.Vendor_Name, v.Item_Supplied, v."Avg_Lead_Time (days)", v.Cost_Per_Item
order by avg_leadtime desc;

"""
vendor_summary = pd.read_sql_query(query4_updated, conn)
print(vendor_summary)


  Vendor_ID       Vendor_Name  Item_Supplied  avg_leadtime  Cost_Per_Item  \
0      V002      EquipMed Co.     Ventilator            30        20000.0   
1      V003  HealthTools Ltd.  X-ray Machine            15         5000.0   
2      V001  MedSupplies Inc.  Surgical Mask             5            0.5   

   Items_Supplied_Count  Total_Stock_Value Next_Delivery_Date  
0                   156       7.379300e+09         2024-10-15  
1                   156       1.918670e+09         2024-10-05  
2                   188       2.383115e+05         2024-10-03  


EquipMed Co. (V002) supplies high-cost ventilators with the longest average lead time of 30 days and holds the highest total stock value (~$7.38 billion), indicating significant exposure to delivery delays and cost risk. HealthTools Ltd. (V003) and MedSupplies Inc. (V001) supply essential equipment and consumables with shorter lead times and lower total stock values.

**Key Insight:**
Vendor reliability varies substantially. Vendors supplying critical, high-value equipment with long lead times pose the greatest operational risk.

**Business Suggestion:**

- Prioritize renegotiations or strategic partnerships with vendors like EquipMed Co. to improve delivery timelines.

- Explore diversifying suppliers or adjusting inventory buffers for critical, slow-to-deliver items to mitigate risks.

### Business Question 5:
What are the hidden inefficiencies or anomalies in hospital operations?

#### Why this matters:
Spotting unusual expense spikes, excessive staff overtime, or abnormal equipment usage helps hospital management identify inefficiencies, control costs, and improve operational processes.


In [56]:
query5_overtime = """
select Staff_Type, Current_Assignment, avg(Overtime_Hours) as overtime
from staff_data
group by Staff_Type, Current_Assignment
order by overtime desc
"""
overtime_hotspots = pd.read_sql_query(query5_overtime, conn)
print(overtime_hotspots)

   Staff_Type Current_Assignment  overtime
0     Surgeon                 ER  2.147541
1  Technician       General Ward  2.103448
2  Technician        ICU Surgery  2.020833
3       Nurse                 ER  2.014286
4  Technician                 ER  2.000000
5       Nurse        ICU Surgery  1.951220
6     Surgeon       General Ward  1.884615
7       Nurse       General Ward  1.790323
8     Surgeon        ICU Surgery  1.693878


In [59]:
query5_equipment = """
select Equipment_Used, Procedure_Performed, count(*) as times_used
from patient_data
group by Equipment_Used, Procedure_Performed 
order by times_used desc
"""
equip_usage = pd.read_sql_query(query5_equipment, conn)
print(equip_usage.head(20))

    Equipment_Used Procedure_Performed  times_used
0    X-ray Machine         Chest X-ray          52
1      MRI Machine                 MRI          50
2   Surgical Table         Chest X-ray          49
3   Surgical Table        Appendectomy          47
4      MRI Machine        Appendectomy          42
5    X-ray Machine        Appendectomy          42
6   Surgical Table                 MRI          41
7    X-ray Machine                 MRI          40
8    X-ray Machine          Blood Test          39
9      MRI Machine         Chest X-ray          37
10     MRI Machine          Blood Test          32
11  Surgical Table          Blood Test          29


Surgeons working in the ER accumulate the most overtime on average (~2.15 hours), followed closely by technicians in the General Ward and ICU Surgery units. Equipment like X-ray and MRI machines are heavily used across multiple procedures, with the X-ray machine involved in 52 Chest X-ray procedures and the MRI machine appearing in 50 MRI procedures, indicating high utilization.

**Key Insight:**
High overtime among specific staff types and intense usage of critical equipment suggests workload imbalances and potential resource strain, risking staff burnout and equipment wear.

**Business Suggestion:**

- Adjust staff scheduling to balance workloads and reduce overtime, especially for surgeons and technicians.

- Plan preventive maintenance and consider increasing equipment capacity or availability to avoid bottlenecks caused by overuse.


# Conclusion

This SQL analysis provides actionable insights into hospital supply chain management and operational performance:

- Identifying inventory risks allows proactive measures to prevent critical stockouts.  
- Expense pattern analysis highlights key cost drivers and seasonal expenditure spikes.  
- Linking patient care complexity to staffing needs informs more efficient workforce planning.  
- Evaluating vendor lead times and costs guides strategic supplier management.  
- Detecting operational inefficiencies in staff overtime and equipment usage supports targeted process improvements.

Together, these findings demonstrate how data-driven decision-making using SQL analytics can enhance hospital efficiency, reduce costs, and improve patient care outcomes.

Future work could extend to predictive analytics for demand forecasting and automated real-time monitoring dashboards to alert management about emerging risks.
